# 69-Claim CFR+ Training Path Sanity Check

Compact notebook to isolate whether the catastrophic 69-claim run comes from the evaluator, the old/non-streamed training path, or the new streamed traversal path.

In [ ]:
import gc
import json
from pathlib import Path
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_fitted_return_action_conditioned import ActionConditionedFittedReturnBRTrainer
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.approx_br import evaluate_approximate_br
from liars_poker.serialization import load_policy, save_policy

assert torch.cuda.is_available(), 'Run this on the CUDA VM.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')

spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(spec).claims) == 69

RUN_ROOT = REPO_ROOT / 'artifacts' / 'cfr_plus_69_claim_path_sanity'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
KNOWN_GOOD_RUN_DIR = REPO_ROOT / 'artifacts' / 'cfr_plus_approx_reference_runs' / 'r6_s4_h4_hp2ptfhq_ss___20260622-152749'
KNOWN_GOOD_SNAPSHOT_MINUTES = 60

TRAIN_ITERATIONS = 1500          # about 15m on the old 256-traversal/cap16 path; adjust if your VM is very different
TRAVERSALS_PER_PLAYER = 256
BR_MINUTES = 5
BR_SEED = 17

print('repo:', REPO_ROOT)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))
print('run root:', RUN_ROOT)


In [ ]:
def json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    raise TypeError(type(obj).__name__)


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, default=json_default) + '\n')


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def slugify(text: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')


def closest_known_good_snapshot(run_dir: Path, target_minutes: float) -> Path:
    candidates = []
    root = run_dir / 'evaluations' / 'policy_snapshots' / 'snapshots'
    for metadata in root.rglob('metadata.json'):
        match = re.search(r'train_(\d+(?:\.\d+)?)_iter_', str(metadata.parent))
        if match:
            train_s = float(match.group(1))
            candidates.append((abs(train_s / 60.0 - target_minutes), train_s / 60.0, metadata.parent))
    if not candidates:
        fallback = run_dir / 'policy'
        if (fallback / 'metadata.json').exists():
            return fallback
        raise FileNotFoundError(f'No policy snapshots found under {run_dir}')
    _, train_min, policy_dir = min(candidates, key=lambda item: item[0])
    print(f'known-good snapshot: {policy_dir} ({train_min:.2f}m)')
    return policy_dir


OLD_PATH_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': 7,
    'regret_buffer_capacity': 1_000_000,
    'strategy_buffer_capacity': 1_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'strategy_weighting': 'linear',
    'regret_positive_weight': 0.5,
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 64,
    'traverser_action_sample_count': 16,
    'traverser_action_sample_fraction': None,
    'traverser_action_full_first': False,
    'traverser_action_sample_schedule': None,
    'traverser_action_priority_count': 0,
    'traverser_action_baseline': 'none',
    'traverser_action_sample_mode': 'random',
    'traversal_streaming': False,
    'traversal_live_row_budget': None,
    'traverser_action_chunk_size': None,
    'traversal_record_flush_size': 131_072,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': device,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
    'seed': BR_SEED,
}


In [ ]:
def run_br(label: str, policy_dir: Path, *, minutes: float = BR_MINUTES) -> dict:
    policy, loaded_spec = load_policy(str(policy_dir))
    if loaded_spec != spec:
        raise ValueError(f'Spec mismatch for {policy_dir}: {loaded_spec}')

    trainer = ActionConditionedFittedReturnBRTrainer(loaded_spec, policy, **BR_TRAINER_KWARGS)
    out_dir = RUN_ROOT / slugify(label) / 'br'
    out_dir.mkdir(parents=True, exist_ok=True)

    measured_s = 0.0
    next_print = 60.0
    while measured_s < 60.0 * minutes:
        start = time.perf_counter()
        record = trainer.run_iteration(episodes_per_role=4096, rollout_batch_size=1024)
        iter_s = time.perf_counter() - start
        measured_s += iter_s
        append_jsonl(out_dir / 'training.jsonl', {
            'label': label,
            'iteration': trainer.iteration,
            'measured_responder_s': measured_s,
            'iteration_s': iter_s,
            'record': record,
        })
        if measured_s >= next_print:
            print(f'  {label}: BR={measured_s/60:.1f}m iter={trainer.iteration} iter_s={iter_s:.3f}')
            next_print += 60.0

    metrics = evaluate_approximate_br(
        trainer,
        episodes_per_role=200_000,
        rollout_batch_size=8192,
    )
    row = {
        'label': label,
        'policy_dir': str(policy_dir),
        'responder_training_min': measured_s / 60.0,
        **metrics,
    }
    append_jsonl(RUN_ROOT / 'br_results.jsonl', row)
    display(pd.DataFrame([row]))
    del trainer, policy
    gc.collect()
    torch.cuda.empty_cache()
    return row


def run_training_case(label: str, trainer_kwargs: dict, *, iterations: int = TRAIN_ITERATIONS) -> tuple[Path, dict]:
    case_dir = RUN_ROOT / slugify(label)
    policy_dir = case_dir / 'policy'
    summary_path = case_dir / 'summary.json'
    if summary_path.exists() and (policy_dir / 'metadata.json').exists():
        print('already trained:', label)
        return policy_dir, json.loads(summary_path.read_text(encoding='utf-8'))

    case_dir.mkdir(parents=True, exist_ok=True)
    (case_dir / 'trainer_kwargs.json').write_text(
        json.dumps({k: (str(v) if k == 'device' else v) for k, v in trainer_kwargs.items()}, indent=2, default=json_default),
        encoding='utf-8',
    )
    trainer = DeepCFRPlusTrainer(spec, **trainer_kwargs)
    measured_s = 0.0
    for local_iter in range(1, iterations + 1):
        start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
        iter_s = time.perf_counter() - start
        measured_s += iter_s
        row = {
            'label': label,
            'local_iteration': local_iter,
            'trainer_iteration': trainer.iteration,
            'measured_training_s': measured_s,
            'iteration_s': iter_s,
            'timing': record.get('timing', {}),
            'action_sampling': record.get('action_sampling', {}),
            'regret_loss': record.get('regret_loss'),
            'strategy_loss': record.get('strategy_loss'),
            'new_regret_records': record.get('new_regret_records'),
            'new_strategy_records': record.get('new_strategy_records'),
        }
        append_jsonl(case_dir / 'training.jsonl', row)
        if local_iter == 1 or local_iter % 100 == 0:
            sampling = row['action_sampling']
            print(
                f'{label}: iter={local_iter}/{iterations} train={measured_s/60:.1f}m '
                f'iter_s={iter_s:.3f} edges={sampling.get("claim_edge_fraction", float("nan")):.3f} '
                f'chunks={sampling.get("streamed_edge_chunks", 0)} splits={sampling.get("streamed_row_splits", 0)}'
            )

    save_policy(trainer.average_policy(), str(policy_dir))
    summary = {
        'label': label,
        'iterations': trainer.iteration,
        'measured_training_min': measured_s / 60.0,
        'policy_dir': str(policy_dir),
        'trainer_kwargs': {k: (str(v) if k == 'device' else v) for k, v in trainer_kwargs.items()},
    }
    summary_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding='utf-8')
    display(pd.DataFrame([summary]))
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return policy_dir, summary


## 1. Known-Good Snapshot BR

This checks whether the current BR/evaluation path still gives a sane number on an old snapshot.

In [ ]:
known_good_policy_dir = closest_known_good_snapshot(KNOWN_GOOD_RUN_DIR, KNOWN_GOOD_SNAPSHOT_MINUTES)
known_good_br = run_br('known-good old snapshot', known_good_policy_dir, minutes=BR_MINUTES)


## 2. Old Non-Streamed Training Path

This reproduces the known-good training configuration as closely as possible: random cap16, no CALL baseline, no streaming, 256 traversals/player.

In [ ]:
old_policy_dir, old_summary = run_training_case('old path cap16 random nonstreamed', OLD_PATH_KWARGS)
old_br = run_br('old path cap16 random nonstreamed', old_policy_dir, minutes=BR_MINUTES)


## 3. Streamed Path, No Effective Splitting

Same CFR+ parameters as the old path, but forces the new streamed traversal implementation with huge chunk/row budgets.

In [ ]:
stream_no_split_kwargs = dict(OLD_PATH_KWARGS)
stream_no_split_kwargs.update({
    'traversal_streaming': True,
    'traversal_live_row_budget': 1_000_000_000,
    'traverser_action_chunk_size': 1_000_000_000,
    'traversal_record_flush_size': 1_000_000_000,
})
stream_no_split_policy_dir, stream_no_split_summary = run_training_case('streamed cap16 random no effective split', stream_no_split_kwargs)
stream_no_split_br = run_br('streamed cap16 random no effective split', stream_no_split_policy_dir, minutes=BR_MINUTES)


## 4. Streamed Path, Heavy Splitting

Same CFR+ parameters again, but with very small streaming/chunk budgets so the streamed path is exercised aggressively.

In [ ]:
stream_heavy_kwargs = dict(OLD_PATH_KWARGS)
stream_heavy_kwargs.update({
    'traversal_streaming': True,
    'traversal_live_row_budget': 64,
    'traverser_action_chunk_size': 32,
    'traversal_record_flush_size': 4096,
})
stream_heavy_policy_dir, stream_heavy_summary = run_training_case('streamed cap16 random heavy split', stream_heavy_kwargs)
stream_heavy_br = run_br('streamed cap16 random heavy split', stream_heavy_policy_dir, minutes=BR_MINUTES)


## Summary

In [ ]:
results = pd.DataFrame(read_jsonl(RUN_ROOT / 'br_results.jsonl'))
if not results.empty:
    display(
        results[[
            'label', 'responder_training_min', 'p_first', 'p_second',
            'exploitability_estimate', 'exploitability_lower_bound', 'policy_dir'
        ]].sort_values('exploitability_estimate')
    )
    ax = results.sort_values('exploitability_estimate').plot.barh(
        x='label',
        y=['exploitability_estimate', 'exploitability_lower_bound'],
        figsize=(10, 4),
    )
    ax.set_xlabel('Approximate exploitability')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
